In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/img_highres.zip" -d "/content/fashion_dataset"

In [ ]:
pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import transforms
from tqdm import tqdm
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import faiss

In [ ]:
class MyFashionDataset(Dataset):
  def __init__(self, metadata, transform):
    self.metadata = metadata
    self.transform = transform

  def __getitem__(self, idx):
    row = self.metadata.iloc[idx]
    image = Image.open(row.image_path).convert('RGB')
    return self.transform(image), row.item_id

  def __len__(self):
    return len(self.metadata)


class Embedder(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT
        backbone = models.resnet50(weights=weights)
        backbone.fc = nn.Identity()
        self.resnet = backbone
        for param in self.resnet.parameters():
            param.requires_grad = False
        for param in self.resnet.layer3.parameters():
            param.requires_grad = True
        for param in self.resnet.layer4.parameters():
            param.requires_grad = True
        self.fc = nn.Linear(2048, embedding_dim)
        self._frozen_sections = [self.resnet.conv1, self.resnet.bn1, self.resnet.layer1, self.resnet.layer2]

    def forward(self, x):
        x = self.resnet(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return torch.nn.functional.normalize(x, dim=-1)

    def train(self, mode=True):
        """requires_grad=False does not stop BatchNorm running stats from drifting --
        that's controlled by .training, not requires_grad. Verified: without this
        override, all 24 BN layers in the "frozen" sections drift anyway during training."""
        super().train(mode)
        if mode:
            for section in self._frozen_sections:
                for m in section.modules():
                    if isinstance(m, nn.BatchNorm2d):
                        m.eval()
        return self


def batch_hard_triplet_loss(embeddings, item_ids, margin=0.3):
    """
    Vectorized batch-hard mining (Hermans et al., "In Defense of the Triplet Loss").
    Per anchor: hardest positive = same item_id, MAX distance. hardest negative =
    different item_id, MIN distance. Replaces triplet_loss + hard_negative_mining.
    Requires PK-sampled batches (K>=2) -- raises a clear error otherwise.
    """
    device = embeddings.device
    item_ids_np = np.asarray(item_ids)

    dot = embeddings @ embeddings.T
    sq_norms = dot.diag().unsqueeze(0)
    dist_sq = (sq_norms.T + sq_norms - 2 * dot).clamp(min=0)

    same_id = torch.from_numpy(item_ids_np[None, :] == item_ids_np[:, None]).to(device)
    N = embeddings.size(0)
    self_mask = torch.eye(N, dtype=torch.bool, device=device)

    positive_mask = same_id & ~self_mask
    negative_mask = ~same_id

    if not positive_mask.any(dim=1).all():
        missing = (~positive_mask.any(dim=1)).sum().item()
        raise ValueError(
            f"{missing} anchor(s) have no positive in this batch -- check that "
            f"CategoryAwarePKSampler (K>=2) is wired into your DataLoader."
        )

    hardest_positive_dist, _ = dist_sq.masked_fill(~positive_mask, float("-inf")).max(dim=1)
    hardest_negative_dist, _ = dist_sq.masked_fill(~negative_mask, float("inf")).min(dim=1)

    return torch.clamp(hardest_positive_dist - hardest_negative_dist + margin, min=0).mean()


def train(model, dataset, sampler, optimizer, device, num_epochs=3, checkpoint_dir='/content/drive/MyDrive'):
    loader = DataLoader(dataset, batch_sampler=sampler, num_workers=2, pin_memory=True)
    model.to(device)
    print(f'Starting training on {device}')

    for epoch in range(1, num_epochs + 1):
        model.train()  # triggers the BN-freeze-aware override above
        epoch_loss, batch_count = 0.0, 0

        for batch_images, batch_item_ids in tqdm(loader):
            batch_images = batch_images.to(device)
            optimizer.zero_grad()
            embeddings = model(batch_images)
            loss = batch_hard_triplet_loss(embeddings, batch_item_ids, margin=0.3)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            batch_count += 1

        avg_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
        print(f"Epoch {epoch}/{num_epochs} | Average Loss: {avg_loss:.4f}")

        # keep EVERY epoch's checkpoint -- evaluate each against 0.546, don't assume last=best
        checkpoint_path = f'{checkpoint_dir}/embedder_full_train_epoch_{epoch}.pt'
        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(), "avg_loss": avg_loss}, checkpoint_path)

    return model

In [ ]:
import random
from collections import defaultdict
from typing import Sequence, Optional
from torch.utils.data import Sampler


class CategoryAwarePKSampler(Sampler):
    """
    Batch sampler for triplet/metric learning on category-organized datasets (e.g. DeepFashion).

    Each yielded batch contains P item_ids drawn from a single randomly chosen category,
    with K images per item_id (batch size = P * K). Restricting each batch to one category
    biases negatives toward visually similar items (e.g. jacket vs. jacket rather than
    jacket vs. shoe), producing harder negatives for triplet mining than uniform random
    sampling across the whole dataset would.

    Sampling semantics (matches pytorch-metric-learning's MPerClassSampler and
    torchreid's RandomIdentitySampler convention):
      - item_ids with >= K images: K sampled WITHOUT replacement each batch.
      - item_ids with <  K images: K sampled WITH replacement (repeats allowed), so no
        item_id is ever discarded just for having too few images.
      - categories with < P distinct item_ids: skipped entirely (can't form a valid
        batch) — a warning is printed at init so this is visible, not silent.

    Args:
        labels: sequence of item_id per dataset index (len == len(dataset)).
        categories: sequence of category per dataset index, aligned with `labels`.
        P: number of distinct item_ids per batch.
        K: number of images per item_id per batch.
        batches_per_epoch: batches to yield per epoch. Defaults to
            (number of unique valid item_ids) // P — roughly one pass over all
            identities per epoch, matching standard PK-sampler convention.
        weighted_by_category_size: if True, categories with more eligible item_ids are
            chosen proportionally more often. If False (default), every valid category
            is equally likely per batch regardless of size — meaning a 5-item category
            and a 3000-item category get sampled equally often. Default is False to
            match the simplest published convention; set True if you want epoch
            composition to reflect your dataset's real category balance.
    """
    def __init__(self, labels, categories, P=8, K=4, batches_per_epoch=None, weighted_by_category_size=False):
        if len(labels) != len(categories):
            raise ValueError(f"labels ({len(labels)}) and categories ({len(categories)}) must be the same length")
        if P < 2:
            raise ValueError("P must be >= 2 (need at least one other identity to form a negative)")
        if K < 2:
            raise ValueError("K must be >= 2 (need at least one positive pair per identity)")
        if batches_per_epoch is not None and batches_per_epoch < 1:
            raise ValueError("batches_per_epoch must be >= 1")

        self.P, self.K = P, K
        self.weighted_by_category_size = weighted_by_category_size

        label_to_idx = defaultdict(list)
        label_to_categories = defaultdict(set)  # a label CAN legitimately belong to >1 category
        for idx, (lab, cat) in enumerate(zip(labels, categories)):
            label_to_idx[lab].append(idx)
            label_to_categories[lab].add(cat)

        multi_cat = {lab: cats for lab, cats in label_to_categories.items() if len(cats) > 1}
        if multi_cat:
            example = next(iter(multi_cat.items()))
            print(f"[CategoryAwarePKSampler] Note: {len(multi_cat)} item_id(s) are tagged under more "
                  f"than one category (e.g. {example[0]!r}: {example[1]}). Each is treated as eligible "
                  f"for every category it's tagged with — not an error.")
            max_cats = max(len(c) for c in multi_cat.values())
            if max_cats > 2:
                print(f"[CategoryAwarePKSampler] Worth a manual look: at least one item_id spans "
                      f"{max_cats} categories.")

        category_to_labels = defaultdict(set)
        for lab, cats in label_to_categories.items():
            for cat in cats:
                category_to_labels[cat].add(lab)
        category_to_labels = {c: list(labs) for c, labs in category_to_labels.items()}
        #random.sample() or random.choice() chenq kara seti u defdict vra anenq
        self.label_to_idx = dict(label_to_idx)
        self.valid_categories = [c for c, labs in category_to_labels.items() if len(labs) >= P]
        self.category_to_labels = {c: category_to_labels[c] for c in self.valid_categories}

        self.n_categories_total = len(category_to_labels)
        n_dropped = self.n_categories_total - len(self.valid_categories)
        if n_dropped > 0:
            print(f"[CategoryAwarePKSampler] Warning: dropped {n_dropped} / {self.n_categories_total} "
                  f"categories with fewer than P={P} distinct item_ids.")
        if not self.valid_categories:
            raise ValueError(f"No category has >= {P} distinct item_ids. Lower P or check your data.")

        self._category_weights = [len(self.category_to_labels[c]) for c in self.valid_categories] if weighted_by_category_size else None

        n_unique_ids = sum(len(v) for v in self.category_to_labels.values())
        self.batches_per_epoch = batches_per_epoch or max(1, n_unique_ids // P)

    def _sample_category(self):
        if self._category_weights is not None:
            return random.choices(self.valid_categories, weights=self._category_weights, k=1)[0]
        return random.choice(self.valid_categories)

    def __iter__(self):
        for _ in range(self.batches_per_epoch):
            cat = self._sample_category()
            batch_ids = random.sample(self.category_to_labels[cat], self.P)
            batch = []
            for item_id in batch_ids:
                idxs = self.label_to_idx[item_id]
                batch.extend(random.sample(idxs, self.K) if len(idxs) >= self.K else random.choices(idxs, k=self.K))
            yield batch

    def __len__(self):
        return self.batches_per_epoch

    def __repr__(self):
        return (f"CategoryAwarePKSampler(P={self.P}, K={self.K}, batch_size={self.P*self.K}, "
                f"batches_per_epoch={self.batches_per_epoch}, valid_categories={len(self.valid_categories)}/{self.n_categories_total})")

In [ ]:
def item_id_train_test_split(metadata, test_size=0.2, random_state=42):
    """
    Splits by item_id so no single item's images ever appear in both train and test --
    including items tagged under more than one category. Each item_id is assigned to
    train or test exactly once; its most common category is used only to decide which
    stratum it counts toward, not to make a separate split decision per category.
    """
    rng = np.random.RandomState(random_state)
    item_to_category = metadata.groupby("item_id")["clothing_category"].agg(lambda s: s.value_counts().idxmax())

    train_ids, test_ids = set(), set()
    for cat, ids_in_cat in item_to_category.groupby(item_to_category):
        unique_ids = np.asarray(ids_in_cat.index, dtype=object)
        rng.shuffle(unique_ids)
        if len(unique_ids) < 2:
            train_ids.update(unique_ids)
            continue
        n_test = max(1, int(round(len(unique_ids) * test_size)))
        test_ids.update(unique_ids[:n_test])
        train_ids.update(unique_ids[n_test:])

    train_df = metadata[metadata.item_id.isin(train_ids)].reset_index(drop=True)
    test_df = metadata[metadata.item_id.isin(test_ids)].reset_index(drop=True)
    return train_df, test_df

In [ ]:
metadata_init = pd.read_csv('original_metadata.csv')
print(metadata_init.shape)
metadata_init.head()

(66464, 4)


,gender,clothing_category,item_id,image_path
0,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...
1,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...
2,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...
3,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...
4,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...


In [ ]:
old_prefix = '/mnt/c/Users/User/Downloads/img_highres/'
new_prefix = '/content/fashion_dataset/'

In [ ]:
metadata_init = metadata_init[~metadata_init.image_path.str.contains('segment')]
metadata = metadata_init[~metadata_init.item_id.isin(['id_00003615',
'id_00006951',
'id_00004776',
'id_00004850',
'id_00007573',
'id_00000773',
'id_00006128',
'id_00006574',
'id_00001453',
'id_00001995',
'id_00002205',
'id_00003020'])]
print(metadata.shape)
metadata.head()

(52700, 4)


,gender,clothing_category,item_id,image_path
0,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...
2,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...
4,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...
5,MEN,Denim,id_00000080,/mnt/c/Users/User/Downloads/img_highres/MEN/De...
6,MEN,Denim,id_00000089,/mnt/c/Users/User/Downloads/img_highres/MEN/De...


In [ ]:
metadata['image_path'] = metadata['image_path'].str.replace(old_prefix, new_prefix)

/tmp/ipykernel_733/3938890495.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['image_path'] = metadata['image_path'].str.replace(old_prefix, new_prefix)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0), ratio=(3/4, 4/3)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])


In [ ]:
train_df, test_df = item_id_train_test_split(metadata, test_size=0.2)


In [ ]:
# full traini masy
train_dataset = MyFashionDataset(metadata.reset_index(drop=True), train_transform)
sampler = CategoryAwarePKSampler(labels=metadata.item_id.tolist(),
                                  categories=metadata.clothing_category.tolist(), P=8, K=4)

model = Embedder(embedding_dim=512).to(device)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

train(model, train_dataset, sampler, optimizer, device, num_epochs=6)

[CategoryAwarePKSampler] Note: 99 item_id(s) are tagged under more than one category (e.g. 'id_00000094': {'Suiting', 'Jackets_Vests'}). Each is treated as eligible for every category it's tagged with — not an error.
Starting training on cuda


100%|██████████| 1008/1008 [09:07<00:00,  1.84it/s]


Epoch 1/6 | Average Loss: 0.3167


100%|██████████| 1008/1008 [08:59<00:00,  1.87it/s]


Epoch 2/6 | Average Loss: 0.2431


...

100%|██████████| 1008/1008 [08:55<00:00,  1.88it/s]


Epoch 5/6 | Average Loss: 0.1473


100%|██████████| 1008/1008 [08:55<00:00,  1.88it/s]


Epoch 6/6 | Average Loss: 0.1294


Embedder(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          

In [ ]:
# traini masy
train_dataset = MyFashionDataset(train_df.reset_index(drop=True), train_transform)
sampler = CategoryAwarePKSampler(labels=train_df.item_id.tolist(),
                                  categories=train_df.clothing_category.tolist(), P=8, K=4)

model = Embedder(embedding_dim=512).to(device)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

train(model, train_dataset, sampler, optimizer, device, num_epochs=6)

In [ ]:
@torch.no_grad()
def extract_embeddings(model, dataset, device, batch_size=64):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    all_embeddings = []
    all_item_ids = []

    for batch_images, batch_item_ids in tqdm(loader):
        batch_images = batch_images.to(device)
        embeddings = model(batch_images)
        all_embeddings.append(embeddings.cpu().numpy())
        all_item_ids.extend(batch_item_ids)

    return np.concatenate(all_embeddings, axis=0), all_item_ids


In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
model = Embedder()
checkpoint = torch.load('/content/drive/MyDrive/embedder_train_epoch_6 (1).pt', map_location=device)
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()

Embedder(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          

In [ ]:
embs_full, labs_full = extract_embeddings(model, MyFashionDataset(metadata.reset_index(drop=True), transform=eval_transform), device)


100%|██████████| 824/824 [10:28<00:00,  1.31it/s]


In [ ]:
np.save('embs_full.npy', embs_full)
np.save('labs_full.npy', labs_full)
from google.colab import files
files.download('embs_full.npy')
files.download('labs_full.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
embs_train, labs_train = extract_embeddings(model, MyFashionDataset(train_df.reset_index(drop=True), transform=eval_transform), device)
embs_test, labs_test = extract_embeddings(model, MyFashionDataset(test_df.reset_index(drop=True), transform=eval_transform), device)

100%|██████████| 164/164 [02:05<00:00,  1.30it/s]


In [ ]:
np.save('embs_train.npy', embs_train)
np.save('embs_test.npy', embs_test)
np.save('labs_train.npy', labs_train)
np.save('labs_test.npy', labs_test)

In [ ]:
from google.colab import files
files.download('embs_train.npy')
files.download('labs_train.npy')
files.download('embs_test.npy')
files.download('labs_test.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
len(embs_full)

52700